# 02 — Modélisation

Repart de `data/processed/matches_features.csv` (généré dans `01_exploration.ipynb`).
Pas besoin de rejouer le chargement/nettoyage/feature engineering — tout est
déjà sauvegardé sur disque.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df_features = pd.read_csv("../data/processed/matches_features.csv", parse_dates=["date"])
print(f"{df_features.shape[0]} matchs, {df_features.shape[1]} colonnes")
df_features.head()

## Nouvelles features : écart de forme domicile/extérieur

Jusqu'ici le modèle voit les stats de forme de chaque équipe **séparément**
(`home_goals_for_avg_last5`, `away_goals_for_avg_last5`, etc.) — il doit
"deviner" lui-même la comparaison entre les deux colonnes. Un match nul
correspond souvent à un écart de niveau faible entre les deux équipes : on
calcule donc explicitement `diff_<stat> = home_<stat> - away_<stat>` pour
rendre cette comparaison directement disponible.

In [ ]:
diff_base_cols = [
    c[len("home_"):] for c in df_features.columns
    if c.startswith("home_") and ("_avg_last" in c or c.endswith("matches_played_before"))
]

for col in diff_base_cols:
    df_features[f"diff_{col}"] = df_features[f"home_{col}"] - df_features[f"away_{col}"]

print(f"{len(diff_base_cols)} features d'écart ajoutées, ex : {[f'diff_{c}' for c in diff_base_cols[:3]]}")

## Split train / validation / test (chronologique, 70/15/15)

Repris tel quel depuis `01_exploration.ipynb`.

In [ ]:
df_model = df_features.sort_values("date").reset_index(drop=True)

df_train, df_temp = train_test_split(df_model, test_size=0.30, shuffle=False)
df_val, df_test = train_test_split(df_temp, test_size=0.50, shuffle=False)

for nom, d in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    print(f"{nom:11s} : {len(d):5d} matchs   ({d['date'].min().date()} -> {d['date'].max().date()})")

## Préparation de X (features) et y (cible)

Repris tel quel depuis `01_exploration.ipynb`.

In [ ]:
def add_target(df):
    """Recalcule le résultat du match (H/D/A) à partir des buts."""
    df = df.copy()
    conditions = [
        df["full_time_home_goals"] > df["full_time_away_goals"],
        df["full_time_home_goals"] == df["full_time_away_goals"],
    ]
    df["result"] = np.select(conditions, ["H", "D"], default="A")
    return df

df_train = add_target(df_train)
df_val = add_target(df_val)
df_test = add_target(df_test)

feature_cols = [
    c for c in df_model.columns
    if "_avg_last" in c
    or c.endswith("matches_played_before")
    or c.endswith("days_since_last_match")
    or "h2h" in c
    or c.endswith("elo_before")
]

# Les moyennes head-to-head sont NaN dès qu'il n'y a aucune confrontation
# directe antérieure dans le dataset -> bien plus fréquent que pour les
# autres features (deux équipes précises se recroisent rarement). On les
# remplace par 0 (valeur neutre : ni domination ni domination adverse)
# plutôt que de perdre la ligne -> h2h_matches_played reste à 0, le modèle
# peut donc apprendre à ignorer ces deux colonnes quand il n'y a pas
# d'historique. Le rating Elo n'a jamais de NaN (valeur par défaut 1500).
h2h_avg_cols = [c for c in feature_cols if "h2h_points_avg" in c or "h2h_goal_diff_avg" in c]

def make_xy(df):
    df_clean = df.copy()
    df_clean[h2h_avg_cols] = df_clean[h2h_avg_cols].fillna(0)
    df_clean = df_clean.dropna(subset=feature_cols)
    return df_clean[feature_cols], df_clean["result"]

X_train, y_train = make_xy(df_train)
X_val, y_val = make_xy(df_val)
X_test, y_test = make_xy(df_test)

for nom, d, X in [("Train", df_train, X_train), ("Validation", df_val, X_val), ("Test", df_test, X_test)]:
    print(f"{nom:11s} : {X.shape[0]:5d} matchs gardés sur {len(d)} ({len(d) - X.shape[0]} retirés pour NaN)")

## Modèle 1 : Random Forest sans limite (référence)

Reproduit le résultat déjà obtenu dans `01_exploration.ipynb` : accuracy
train = 1.000 (sur-apprentissage), validation = 0.522. On garde ce résultat
comme point de comparaison pour juger si la régularisation qui suit aide
vraiment.

In [ ]:
baseline_class = y_train.value_counts().idxmax()
baseline_accuracy = (y_val == baseline_class).mean()
print(f"Baseline (toujours prédire '{baseline_class}') : {baseline_accuracy:.3f} d'accuracy sur validation\n")

rf_v1 = RandomForestClassifier(n_estimators=200, random_state=42)
rf_v1.fit(X_train, y_train)

print(f"Accuracy train      : {accuracy_score(y_train, rf_v1.predict(X_train)):.3f}")
print(f"Accuracy validation : {accuracy_score(y_val, rf_v1.predict(X_val)):.3f}")

## Modèle 2 : Random Forest régularisé

Deux paramètres pour empêcher le modèle de mémoriser le train par cœur :
- `max_depth` : profondeur maximale de chaque arbre — l'empêche de créer une
  branche ultra-spécifique pour un seul match.
- `min_samples_leaf` : nombre minimum de matchs qu'une feuille doit regrouper
  — force chaque décision finale à s'appuyer sur plusieurs cas, pas un seul.

On teste plusieurs combinaisons et on choisit celle qui a la meilleure
accuracy sur **validation** (jamais sur test).

In [ ]:
results = []

for max_depth in [4, 6, 8, 10, None]:
    for min_samples_leaf in [1, 5, 20, 50]:
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=42,
        )
        model.fit(X_train, y_train)
        train_acc = accuracy_score(y_train, model.predict(X_train))
        val_acc = accuracy_score(y_val, model.predict(X_val))
        results.append({
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
        })

df_results = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
df_results.head(10)

### Meilleure combinaison retenue

On reprend la ligne avec la meilleure `val_accuracy` du tableau ci-dessus,
on réentraîne ce modèle, et on regarde son rapport détaillé sur validation
(notamment si la classe "D" reste ignorée).

In [ ]:
best_params = df_results.iloc[0]
print(f"Meilleure combinaison : max_depth={best_params['max_depth']}, min_samples_leaf={int(best_params['min_samples_leaf'])}")

rf_v2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=None if pd.isna(best_params["max_depth"]) else int(best_params["max_depth"]),
    min_samples_leaf=int(best_params["min_samples_leaf"]),
    random_state=42,
)
rf_v2.fit(X_train, y_train)

print(f"\nAccuracy train      : {accuracy_score(y_train, rf_v2.predict(X_train)):.3f}")
print(f"Accuracy validation : {accuracy_score(y_val, rf_v2.predict(X_val)):.3f}")
print(f"(baseline           : {baseline_accuracy:.3f})")

print("\nRapport détaillé (validation) :")
print(classification_report(y_val, rf_v2.predict(X_val)))

## Modèle 3 : `class_weight="balanced"` — récupérer les nuls

`class_weight="balanced"` pénalise plus fortement les erreurs sur les classes
minoritaires (ici "D") pendant l'entraînement, pour empêcher le modèle de les
ignorer.

On refait la même recherche que pour le modèle 2, mais on sélectionne cette
fois la meilleure combinaison sur le **f1-score macro** plutôt que l'accuracy
(cf. explication ci-dessus).

In [7]:
from sklearn.metrics import f1_score

results_balanced = []

for max_depth in [4, 6, 8, 10, None]:
    for min_samples_leaf in [1, 5, 20, 50]:
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42,
        )
        model.fit(X_train, y_train)
        val_preds = model.predict(X_val)
        results_balanced.append({
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "val_accuracy": accuracy_score(y_val, val_preds),
            "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
        })

df_results_balanced = pd.DataFrame(results_balanced).sort_values("val_f1_macro", ascending=False)
df_results_balanced.head(10)

,max_depth,min_samples_leaf,val_accuracy,val_f1_macro
4,6.0,1,0.494477,0.470541
5,6.0,5,0.491163,0.467211
3,4.0,50,0.483063,0.466691
0,4.0,1,0.483063,0.466508
1,4.0,5,0.483063,0.466508
2,4.0,20,0.482695,0.466247
11,8.0,50,0.494109,0.463926
9,8.0,5,0.499264,0.463876
10,8.0,20,0.493741,0.463713
13,10.0,5,0.509941,0.463438


In [8]:
best_balanced = df_results_balanced.iloc[0]
print(f"Meilleure combinaison (f1 macro) : max_depth={best_balanced['max_depth']}, min_samples_leaf={int(best_balanced['min_samples_leaf'])}")

rf_v3 = RandomForestClassifier(
    n_estimators=200,
    max_depth=None if pd.isna(best_balanced["max_depth"]) else int(best_balanced["max_depth"]),
    min_samples_leaf=int(best_balanced["min_samples_leaf"]),
    class_weight="balanced",
    random_state=42,
)
rf_v3.fit(X_train, y_train)

print(f"\nAccuracy train      : {accuracy_score(y_train, rf_v3.predict(X_train)):.3f}")
print(f"Accuracy validation : {accuracy_score(y_val, rf_v3.predict(X_val)):.3f}")
print(f"(baseline           : {baseline_accuracy:.3f})")

print("\nRapport détaillé (validation) :")
print(classification_report(y_val, rf_v3.predict(X_val)))

Meilleure combinaison (f1 macro) : max_depth=6.0, min_samples_leaf=1

Accuracy train      : 0.536
Accuracy validation : 0.494
(baseline           : 0.430)

Rapport détaillé (validation) :
              precision    recall  f1-score   support

           A       0.49      0.57      0.53       838
           D       0.31      0.28      0.29       710
           H       0.61      0.57      0.59      1168

    accuracy                           0.49      2716
   macro avg       0.47      0.47      0.47      2716
weighted avg       0.49      0.49      0.49      2716



## Modèle 4 : mêmes réglages que v3, + features d'écart

`feature_cols` capte automatiquement les nouvelles colonnes `diff_*` (elles
contiennent `_avg_last` ou finissent par `matches_played_before`, comme les
colonnes `home_*`/`away_*`). Même recherche que pour le modèle 3
(`class_weight="balanced"`, sélection par f1-macro), pour comparer à
features égales sauf l'ajout des écarts.

In [ ]:
print(f"{len(feature_cols)} features (dont {sum('diff_' in c for c in feature_cols)} d'écart)")

results_v4 = []

for max_depth in [4, 6, 8, 10, None]:
    for min_samples_leaf in [1, 5, 20, 50]:
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42,
        )
        model.fit(X_train, y_train)
        val_preds = model.predict(X_val)
        results_v4.append({
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "val_accuracy": accuracy_score(y_val, val_preds),
            "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
        })

df_results_v4 = pd.DataFrame(results_v4).sort_values("val_f1_macro", ascending=False)
df_results_v4.head(10)

In [ ]:
best_v4 = df_results_v4.iloc[0]
print(f"Meilleure combinaison (f1 macro) : max_depth={best_v4['max_depth']}, min_samples_leaf={int(best_v4['min_samples_leaf'])}")

rf_v4 = RandomForestClassifier(
    n_estimators=200,
    max_depth=None if pd.isna(best_v4["max_depth"]) else int(best_v4["max_depth"]),
    min_samples_leaf=int(best_v4["min_samples_leaf"]),
    class_weight="balanced",
    random_state=42,
)
rf_v4.fit(X_train, y_train)

print(f"\nAccuracy train      : {accuracy_score(y_train, rf_v4.predict(X_train)):.3f}")
print(f"Accuracy validation : {accuracy_score(y_val, rf_v4.predict(X_val)):.3f}")
print(f"f1-macro validation : {f1_score(y_val, rf_v4.predict(X_val), average='macro'):.3f}")
print(f"(pour comparaison, v3 sans features d'écart : accuracy 0.494, f1-macro 0.47)")

print("\nRapport détaillé (validation) :")
print(classification_report(y_val, rf_v4.predict(X_val)))

## Modèle 5 : + forme domicile/extérieur séparée

Nécessite d'avoir régénéré `matches_features.csv` depuis `01_exploration.ipynb`
(nouvelles colonnes `*_avg_last5_venue` / `*_avg_last10_venue`). `feature_cols`
les capte automatiquement. Même recherche que pour v3/v4.

In [ ]:
print(f"{len(feature_cols)} features (dont {sum('_venue' in c for c in feature_cols)} de forme domicile/extérieur)")

results_v5 = []

for max_depth in [4, 6, 8, 10, None]:
    for min_samples_leaf in [1, 5, 20, 50]:
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42,
        )
        model.fit(X_train, y_train)
        val_preds = model.predict(X_val)
        results_v5.append({
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "val_accuracy": accuracy_score(y_val, val_preds),
            "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
        })

df_results_v5 = pd.DataFrame(results_v5).sort_values("val_f1_macro", ascending=False)
df_results_v5.head(10)

In [ ]:
best_v5 = df_results_v5.iloc[0]
print(f"Meilleure combinaison (f1 macro) : max_depth={best_v5['max_depth']}, min_samples_leaf={int(best_v5['min_samples_leaf'])}")

rf_v5 = RandomForestClassifier(
    n_estimators=200,
    max_depth=None if pd.isna(best_v5["max_depth"]) else int(best_v5["max_depth"]),
    min_samples_leaf=int(best_v5["min_samples_leaf"]),
    class_weight="balanced",
    random_state=42,
)
rf_v5.fit(X_train, y_train)

print(f"\nAccuracy train      : {accuracy_score(y_train, rf_v5.predict(X_train)):.3f}")
print(f"Accuracy validation : {accuracy_score(y_val, rf_v5.predict(X_val)):.3f}")
print(f"f1-macro validation : {f1_score(y_val, rf_v5.predict(X_val), average='macro'):.3f}")
print(f"(pour comparaison, v3/v4 : accuracy ~0.494, f1-macro ~0.47)")

print("\nRapport détaillé (validation) :")
print(classification_report(y_val, rf_v5.predict(X_val)))

## Modèle 6 : + repos entre matchs

Nécessite d'avoir régénéré `matches_features.csv` (nouvelle colonne
`days_since_last_match`, désormais captée par `feature_cols`). Même
recherche que v3/v4/v5.

In [ ]:
print(f"{len(feature_cols)} features (dont {sum('days_since' in c for c in feature_cols)} de repos entre matchs)")

results_v6 = []

for max_depth in [4, 6, 8, 10, None]:
    for min_samples_leaf in [1, 5, 20, 50]:
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42,
        )
        model.fit(X_train, y_train)
        val_preds = model.predict(X_val)
        results_v6.append({
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "val_accuracy": accuracy_score(y_val, val_preds),
            "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
        })

df_results_v6 = pd.DataFrame(results_v6).sort_values("val_f1_macro", ascending=False)
df_results_v6.head(10)

In [ ]:
best_v6 = df_results_v6.iloc[0]
print(f"Meilleure combinaison (f1 macro) : max_depth={best_v6['max_depth']}, min_samples_leaf={int(best_v6['min_samples_leaf'])}")

rf_v6 = RandomForestClassifier(
    n_estimators=200,
    max_depth=None if pd.isna(best_v6["max_depth"]) else int(best_v6["max_depth"]),
    min_samples_leaf=int(best_v6["min_samples_leaf"]),
    class_weight="balanced",
    random_state=42,
)
rf_v6.fit(X_train, y_train)

print(f"\nAccuracy train      : {accuracy_score(y_train, rf_v6.predict(X_train)):.3f}")
print(f"Accuracy validation : {accuracy_score(y_val, rf_v6.predict(X_val)):.3f}")
print(f"f1-macro validation : {f1_score(y_val, rf_v6.predict(X_val), average='macro'):.3f}")
print(f"(pour comparaison, v5 sans repos entre matchs : accuracy 0.506, f1-macro 0.479)")

print("\nRapport détaillé (validation) :")
print(classification_report(y_val, rf_v6.predict(X_val)))

## Modèle 7 : + historique des confrontations directes (head-to-head)

Nécessite d'avoir régénéré `matches_features.csv`. Les colonnes `h2h_*` sont
désormais captées par `feature_cols`, avec le traitement spécifique des NaN
vu ci-dessus. Même recherche que v3 à v6.

In [ ]:
print(f"{len(feature_cols)} features (dont {sum('h2h' in c for c in feature_cols)} head-to-head)")

results_v7 = []

for max_depth in [4, 6, 8, 10, None]:
    for min_samples_leaf in [1, 5, 20, 50]:
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42,
        )
        model.fit(X_train, y_train)
        val_preds = model.predict(X_val)
        results_v7.append({
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "val_accuracy": accuracy_score(y_val, val_preds),
            "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
        })

df_results_v7 = pd.DataFrame(results_v7).sort_values("val_f1_macro", ascending=False)
df_results_v7.head(10)

In [ ]:
best_v7 = df_results_v7.iloc[0]
print(f"Meilleure combinaison (f1 macro) : max_depth={best_v7['max_depth']}, min_samples_leaf={int(best_v7['min_samples_leaf'])}")

rf_v7 = RandomForestClassifier(
    n_estimators=200,
    max_depth=None if pd.isna(best_v7["max_depth"]) else int(best_v7["max_depth"]),
    min_samples_leaf=int(best_v7["min_samples_leaf"]),
    class_weight="balanced",
    random_state=42,
)
rf_v7.fit(X_train, y_train)

print(f"\nAccuracy train      : {accuracy_score(y_train, rf_v7.predict(X_train)):.3f}")
print(f"Accuracy validation : {accuracy_score(y_val, rf_v7.predict(X_val)):.3f}")
print(f"f1-macro validation : {f1_score(y_val, rf_v7.predict(X_val), average='macro'):.3f}")
print(f"(pour comparaison, v6 sans head-to-head : accuracy 0.511, f1-macro 0.481)")

print("\nRapport détaillé (validation) :")
print(classification_report(y_val, rf_v7.predict(X_val)))

## Modèle 8 : + rating Elo

Dernière feature de la liste. Nécessite d'avoir régénéré
`matches_features.csv`. Même recherche que v3 à v7.

In [ ]:
print(f"{len(feature_cols)} features (dont {sum('elo' in c for c in feature_cols)} Elo)")

results_v8 = []

for max_depth in [4, 6, 8, 10, None]:
    for min_samples_leaf in [1, 5, 20, 50]:
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42,
        )
        model.fit(X_train, y_train)
        val_preds = model.predict(X_val)
        results_v8.append({
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "val_accuracy": accuracy_score(y_val, val_preds),
            "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
        })

df_results_v8 = pd.DataFrame(results_v8).sort_values("val_f1_macro", ascending=False)
df_results_v8.head(10)

In [ ]:
best_v8 = df_results_v8.iloc[0]
print(f"Meilleure combinaison (f1 macro) : max_depth={best_v8['max_depth']}, min_samples_leaf={int(best_v8['min_samples_leaf'])}")

rf_v8 = RandomForestClassifier(
    n_estimators=200,
    max_depth=None if pd.isna(best_v8["max_depth"]) else int(best_v8["max_depth"]),
    min_samples_leaf=int(best_v8["min_samples_leaf"]),
    class_weight="balanced",
    random_state=42,
)
rf_v8.fit(X_train, y_train)

print(f"\nAccuracy train      : {accuracy_score(y_train, rf_v8.predict(X_train)):.3f}")
print(f"Accuracy validation : {accuracy_score(y_val, rf_v8.predict(X_val)):.3f}")
print(f"f1-macro validation : {f1_score(y_val, rf_v8.predict(X_val), average='macro'):.3f}")
print(f"(pour comparaison, v7 sans Elo : accuracy 0.507, f1-macro 0.482)")

print("\nRapport détaillé (validation) :")
print(classification_report(y_val, rf_v8.predict(X_val)))

print("\nImportance des features (top 15) :")
importances = pd.Series(rf_v8.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.head(15))

## Évaluation finale sur le jeu de test

**Première et unique fois qu'on regarde le test.** On prend `rf_v8` tel
quel (dernier modèle entraîné, cumulant toutes les features construites) —
aucun réglage n'est choisi en fonction du résultat obtenu ici, ce serait
retomber dans le même travers que celui qu'on vient d'éviter avec la
validation.

In [74]:
test_preds = rf_v8.predict(X_test)

print(f"Accuracy test : {accuracy_score(y_test, test_preds):.3f}")
print(f"f1-macro test : {f1_score(y_test, test_preds, average='macro'):.3f}")
print(f"(pour référence, sur validation : accuracy {accuracy_score(y_val, rf_v8.predict(X_val)):.3f}, f1-macro {f1_score(y_val, rf_v8.predict(X_val), average='macro'):.3f})")

print("\nRapport détaillé (test) :")
print(classification_report(y_test, test_preds))

Accuracy test : 0.486
f1-macro test : 0.465
(pour référence, sur validation : accuracy 0.507, f1-macro 0.482)

Rapport détaillé (test) :
              precision    recall  f1-score   support

           A       0.49      0.54      0.51       853
           D       0.29      0.32      0.31       664
           H       0.61      0.54      0.58      1159

    accuracy                           0.49      2676
   macro avg       0.47      0.47      0.47      2676
weighted avg       0.50      0.49      0.49      2676



## Modèle 9 : Gradient Boosting

`HistGradientBoostingClassifier` (implémentation moderne et rapide de
scikit-learn, même famille d'algorithme que XGBoost/LightGBM). Il n'a pas de
paramètre `class_weight` intégré comme le Random Forest — on calcule des
poids d'échantillon équivalents avec `compute_sample_weight`, passés à
`fit()` via `sample_weight`.

Réglé **uniquement sur validation**, comme les modèles précédents — le test
reste de côté pour l'instant.

In [75]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)

results_gb = []

for learning_rate in [0.01, 0.05, 0.1]:
    for max_depth in [3, 5, 8, None]:
        model = HistGradientBoostingClassifier(
            learning_rate=learning_rate,
            max_depth=max_depth,
            max_iter=200,
            random_state=42,
        )
        model.fit(X_train, y_train, sample_weight=sample_weight_train)
        val_preds = model.predict(X_val)
        results_gb.append({
            "learning_rate": learning_rate,
            "max_depth": max_depth,
            "val_accuracy": accuracy_score(y_val, val_preds),
            "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
        })

df_results_gb = pd.DataFrame(results_gb).sort_values("val_f1_macro", ascending=False)
df_results_gb.head(10)

KeyboardInterrupt: 

In [ ]:
best_gb = df_results_gb.iloc[0]
print(f"Meilleure combinaison (f1 macro) : learning_rate={best_gb['learning_rate']}, max_depth={best_gb['max_depth']}")

gb_v9 = HistGradientBoostingClassifier(
    learning_rate=best_gb["learning_rate"],
    max_depth=None if pd.isna(best_gb["max_depth"]) else int(best_gb["max_depth"]),
    max_iter=200,
    random_state=42,
)
gb_v9.fit(X_train, y_train, sample_weight=sample_weight_train)

print(f"\nAccuracy train      : {accuracy_score(y_train, gb_v9.predict(X_train)):.3f}")
print(f"Accuracy validation : {accuracy_score(y_val, gb_v9.predict(X_val)):.3f}")
print(f"f1-macro validation : {f1_score(y_val, gb_v9.predict(X_val), average='macro'):.3f}")
print(f"(pour comparaison, rf_v8 sur validation : accuracy 0.507, f1-macro 0.482)")

print("\nRapport détaillé (validation) :")
print(classification_report(y_val, gb_v9.predict(X_val)))

### Vérification sur le test (2e et dernier regard, exceptionnellement)

In [ ]:
gb_test_preds = gb_v9.predict(X_test)

print(f"Accuracy test (gb_v9) : {accuracy_score(y_test, gb_test_preds):.3f}")
print(f"f1-macro test (gb_v9) : {f1_score(y_test, gb_test_preds, average='macro'):.3f}")
print(f"(pour comparaison, rf_v8 sur test : accuracy 0.486, f1-macro 0.465)")

print("\nRapport détaillé (test, gb_v9) :")
print(classification_report(y_test, gb_test_preds))

Accuracy test (gb_v9) : 0.493
f1-macro test (gb_v9) : 0.461
(pour comparaison, rf_v8 sur test : accuracy 0.486, f1-macro 0.465)

Rapport détaillé (test, gb_v9) :
              precision    recall  f1-score   support

           A       0.48      0.56      0.52       853
           D       0.31      0.25      0.28       664
           H       0.60      0.58      0.59      1159

    accuracy                           0.49      2676
   macro avg       0.46      0.46      0.46      2676
weighted avg       0.49      0.49      0.49      2676



## Sauvegarde du modèle final retenu

`rf_v8` conservé (meilleur f1-macro, meilleure détection des nuls que
`gb_v9`). On sauvegarde aussi `feature_cols` (l'ordre et la liste exacte des
colonnes attendues en entrée) — indispensable pour réutiliser le modèle plus
tard sans se tromper sur les features à fournir.

In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(rf_v8, "../models/rf_v8.joblib")
joblib.dump(feature_cols, "../models/feature_cols.joblib")

print("Modèle sauvegardé -> models/rf_v8.joblib")
print("Liste des features sauvegardée -> models/feature_cols.joblib")
print(f"\nPour le recharger plus tard :")
print(f'  rf_v8 = joblib.load("../models/rf_v8.joblib")')
print(f'  feature_cols = joblib.load("../models/feature_cols.joblib")')

Modèle sauvegardé -> models/rf_v8.joblib
Liste des features sauvegardée -> models/feature_cols.joblib

Pour le recharger plus tard :
  rf_v8 = joblib.load("../models/rf_v8.joblib")
  feature_cols = joblib.load("../models/feature_cols.joblib")
